# E3.1 · Translating agentic risk upward

**Function E — AI Governance for Agentic Systems → The BISO, Risk Communicator & CISO Office**  ·  *Security of AI*

Builds on **[E2.9 · Regulator and auditor conversations](https://spbreed.github.io/cyber-commons/lessons/E2.9.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

The board does not want the threat model. It wants to know the exposure, whether it is going up or down, and what decision is being asked of them — in that order, in language that survives being repeated by someone else.

> **At CyberTravels.** The board does not want TripBot's threat model. It wants the exposure, the direction it is moving, and the decision being asked of them.

## 2 · The framework

```
   technical risk                board-usable exposure
   +-------------------+         +--------------------------+
   | prompt injection  |   -->   | exposure: X, trend: down |
   | in the RAG path   |         | decision asked: fund Y   |
   +-------------------+         +--------------------------+

   it has to survive being repeated by someone else, without you
```

Translating agentic risk upward means dropping every mechanism and keeping three
things: **exposure, likelihood, and the decision being requested.**

The failure mode is not using too much jargon. It is presenting *findings* when
the audience needs a *decision*. A board cannot act on "we found prompt
injection in the review agent". It can act on "a critical system can take N
units of unreviewed action, we demonstrated it, and we are asking for X or for
written acceptance".

That last clause matters more than people expect. **Accepting the risk in
writing, with a named owner and a review date, is a legitimate outcome.** Offering
it makes the ask credible, because it shows you are presenting a decision rather
than lobbying for a budget.

## 3 · Demo — compute the three numbers from what the tracks produced

In [ ]:
SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20}

FLEET = {
 "pr-remediation-agent": [("read_file","self",True),
                          ("write_file","project",True),
                          ("deploy","org",False)],
 "claims-triage-agent":  [("read_file","self",True),
                          ("issue_refund","tenant",False)],
 "doc-summariser":       [("read_file","self",True)],
}
GATED = {"pr-remediation-agent": set(), "claims-triage-agent": {"issue_refund"},
         "doc-summariser": set()}

def blast(tools, gated):
    return sum(SCOPE_WEIGHT[s] * (1 if rev else 2)
               for n, s, rev in tools if n not in gated)

exposure = {a: blast(t, GATED[a]) for a, t in FLEET.items()}
print(f"{'agent':24s}{'blast radius':>14}")
print("-" * 40)
for a, b in sorted(exposure.items(), key=lambda kv: -kv[1]):
    print(f"{a:24s}{b:>14}")
total_exposure = sum(exposure.values())
print(f"{'FLEET TOTAL':24s}{total_exposure:>14}")

# likelihood — measured, from C1.2
ATTACKS = [("metadata service", False), ("path traversal", False),
           ("unlisted egress", True), ("denied tool", False)]
asr = sum(1 for _, through in ATTACKS if through) / len(ATTACKS)
print(f"\nred-team attack success rate (containment surface): {asr:.0%}")

# assurance — from E1.7
REQUIRED = ["AC-1","AC-2","SB-1","SB-2","EV-1","EV-2","DR-1","ST-1"]
EVIDENCED = ["AC-1","AC-2","EV-1","EV-2"]
coverage = len(EVIDENCED) / len(REQUIRED)
print(f"controls currently evidenced: {len(EVIDENCED)}/{len(REQUIRED)} = {coverage:.0%}")

## 4 · Where it breaks — the findings-shaped update

In [ ]:
FINDINGS_UPDATE = """
This quarter the team identified prompt injection in the code review agent,
insufficient scope narrowing in the delegation chain, and gaps in our egress
allowlist. We ran garak and promptfoo against three agents and found a 25%
attack success rate on the containment surface. We recommend prioritising
provenance controls and completing the SPIFFE rollout.
"""
print(FINDINGS_UPDATE)
print("Problems with this, from the audience's side:")
for p in ["no exposure figure — how much can actually happen?",
          "'25% attack success rate' against what, and is that good or bad?",
          "four tool names nobody in the room can evaluate",
          "'recommend prioritising' is not a decision anyone can take",
          "no option to decline, so it reads as lobbying rather than a choice"]:
    print(f"   · {p}")

## 5 · The control — exposure, likelihood, assurance, decision

In [ ]:
def board_translation(tier, exposure, asr, coverage, ask, cost, owner):
    likelihood = ("demonstrated" if asr > 0.2 else
                  "reduced but not eliminated" if asr > 0 else "not demonstrated")
    return f"""
EXPOSURE     A {tier}-tier system can take {exposure} units of unreviewed action.
             (One unit ≈ one irreversible change inside one project.)

LIKELIHOOD   We attacked it. {asr:.0%} of our attack suite succeeded — {likelihood}.
             This is a measurement, not an assessment.

ASSURANCE    {coverage:.0%} of the controls we say we operate are currently
             evidenced. The remainder are untested or their evidence has expired.

DECISION     {ask}
             Cost: {cost}.
             The alternative is to accept the unevidenced portion in writing,
             owned by {owner}, with a review date. Both are acceptable outcomes;
             we need one of them recorded."""

print(board_translation(
    tier="critical", exposure=total_exposure, asr=asr, coverage=coverage,
    ask="Fund continuous control verification for the agent fleet.",
    cost="0.5 FTE for two quarters, no new licences",
    owner="the Chief Operating Officer"))

In [ ]:
# Verify: the translation must contain no mechanism and must offer a choice.
JARGON = ["prompt injection", "spiffe", "garak", "promptfoo", "cwe",
          "provenance", "allowlist", "delegation chain", "token exchange"]
text = board_translation("critical", total_exposure, asr, coverage,
                         "Fund continuous control verification.", "0.5 FTE", "the COO")
found = [j for j in JARGON if j in text.lower()]
print(f"mechanism terms present: {found or 'none'}")
has_choice = "alternative" in text.lower() and "accept" in text.lower()
has_number = str(total_exposure) in text and f"{asr:.0%}" in text
print(f"offers a genuine alternative : {has_choice}")
print(f"carries measured numbers     : {has_number}")
assert not found and has_choice and has_number
print("\nFour facts, no mechanism, and a decision that can go either way.")

## What you just proved

The fleet's exposure totals 46 units, containment ASR is 25%, and control coverage is 50%. The findings-shaped update is shown with five specific problems. The board translation states exposure, likelihood, assurance and a decision, contains no mechanism jargon, carries the measured numbers, and explicitly offers written acceptance as an alternative.

## Your turn

Write these four lines for your highest-tier system. If you cannot fill the likelihood line with a measurement, that is the first thing to fund — an assessment is not a number.

---

**Next → [E3.2 · Governing autonomy rather than approving tools](https://spbreed.github.io/cyber-commons/lessons/E3.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*